# AgroSense — Pakistan Crop Stress Classification

End-to-end walk-through: load the dataset → engineer features → train five classifiers → evaluate and visualise results.

**Dataset:** 1,786 real Sentinel-2 observations · 116 field sites · 4 provinces · 10 seasons (2015–2024)

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score)
from xgboost import XGBClassifier

from feature_engineering import FEATURE_COLS

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/agrosense_crop_stress_dataset.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Class distribution
df['crop_stress_label'].value_counts().plot(kind='bar', color=['#4CAF50','#FF9800','#F44336'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Province breakdown
df.groupby('province')['location_id'].nunique().sort_values().plot(kind='barh')
plt.title('Field Sites per Province')
plt.xlabel('Number of Sites')
plt.tight_layout()

## 3. Feature Analysis

In [ ]:
# NDVI distribution by class
for label, grp in df.groupby('crop_stress_label'):
    grp['ndvi'].plot.kde(label=label)
plt.title('NDVI Distribution by Stress Class')
plt.xlabel('NDVI')
plt.legend()
plt.tight_layout()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5)
plt.title('Spectral Feature Correlation Matrix')
plt.tight_layout()

## 4. Train / Test Split

In [ ]:
le = LabelEncoder()
X = df[FEATURE_COLS].values
y = le.fit_transform(df['crop_stress_label'])
classes = le.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## 5. Train All Five Classifiers

In [ ]:
classifiers = {
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':           XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                       use_label_encoder=False, eval_metric='mlogloss',
                                       random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
    'SVM (RBF)':         SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'k-NN':              KNeighborsClassifier(n_neighbors=5),
}

trained, results = {}, []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    trained[name] = clf
    acc = accuracy_score(y_test, clf.predict(X_test))
    cv_f1 = cross_val_score(clf, X, y, cv=cv, scoring='f1_macro').mean()
    results.append({'Algorithm': name, 'Test Accuracy': acc, 'CV F1 Macro': cv_f1})
    print(f'{name:22s}  Acc={acc:.4f}  CV-F1={cv_f1:.4f}')

results_df = pd.DataFrame(results)
results_df

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, clf) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, clf.predict(X_test), normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9)
plt.suptitle('Normalised Confusion Matrices (Test Set)', y=1.02)
plt.tight_layout()

## 7. Feature Importance

In [ ]:
tree_models = {k: v for k, v in trained.items() if hasattr(v, 'feature_importances_')}
imp = pd.DataFrame({k: v.feature_importances_ for k, v in tree_models.items()},
                    index=FEATURE_COLS).sort_values('Random Forest', ascending=False)

imp.plot(kind='bar', figsize=(10, 4))
plt.title('Feature Importance — Tree Models')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## 8. Best Model — Per-Class Report

In [ ]:
best = trained['Random Forest']
print(classification_report(y_test, best.predict(X_test), target_names=classes))